In [110]:
import numpy as np
import random
import numba as nb

@nb.njit(parallel=True)
def return_crossover_points(empty_matrix, samples, n, k):
    for i in nb.prange(samples):
        empty_matrix[i,1:-1] = np.random.choice(np.arange(1, n-1), size=k, replace=False)
    empty_matrix[:,-1] = n



In [118]:
import time
times_taken = []
for _ in range(100):
    X = np.zeros((100**2, 7))    
    s_time = time.time()
    return_crossover_points(X, 100**2, 250, 5)
    e_time = time.time()

    times_taken.append(e_time - s_time)

In [130]:
@nb.njit(parallel=True, fastmath=True)
def generate_samples_precomputed(rand_pool, samples, k, n):
    empty_matrix = np.zeros((samples, k + 2), dtype=np.int32)
    empty_matrix[:, -1] = n

    for i in nb.prange(samples):
        indices = np.random.randint(0, rand_pool.shape[0], size=k)
        perm = rand_pool[i, indices]
        empty_matrix[i, 1:-1] = np.sort(perm)

    return empty_matrix


Time taken: 13024.87 ms


In [134]:
# Precompute random pool
rand_pool_size = 50_000  # Adjust for balance between memory & speed
rand_pool = np.array([np.random.permutation(n-1)[:7] + 1 for _ in range(rand_pool_size)])

# Timing test
start = time.time()
result = generate_samples_precomputed(rand_pool, 10_000, 5, 250)
end = time.time()
print(f"Time taken: {(end - start) * 1000:.2f} ms")

Time taken: 31.41 ms


In [136]:
from joblib import Parallel, delayed

def generate_single_sample(n, k):
    return np.sort(np.random.choice(np.arange(1, n-1), size=k, replace=False))

def generate_samples_parallel(samples, n, k, n_jobs=-1):
    return np.array(Parallel(n_jobs=n_jobs)(
        delayed(generate_single_sample)(n, k) for _ in range(samples)
    ))



Time taken: 1294.56 ms


In [139]:
# Timing test
start = time.time()
result = generate_samples_parallel(10_000, 250, 5)
end = time.time()
print(f"Time taken: {(end - start) * 1000:.2f} ms")


Time taken: 1178.34 ms
